## **SQL and MySQL Training: Hands-On Training**

### **Notebook 05: Mini Project — Python and SQL Integration**

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![MySQL](https://img.shields.io/badge/MySQL-8.0%2B-orange)](https://www.mysql.com/) [![mysql-connector-python](https://img.shields.io/badge/mysql--connector--python-Latest-green)](https://dev.mysql.com/doc/connector-python/en/) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT) [![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-informational)](https://github.com/prakash-ukhalkar) [![GitHub](https://img.shields.io/badge/GitHub-python--sql--mysql--training-181717?style=flat&logo=github)](https://github.com/prakash-ukhalkar/python-sql-mysql-training)

---

**Author:** `Prakash Ukhalkar`  
**Role:** `Assistant Professor (MCA) | Researcher in Data Science and Machine Learning`

**Notebook Scope:** End-to-end mini project integrating Python with MySQL — schema setup, data population, multi-report generation, and a reusable query function — consolidating all Day 1–4 concepts.

---

This notebook implements a complete Python + SQL mini project on an Employee–Department dataset.

# Python and SQL Training - Day 5
## Mini Project: Python and SQL Integration

**Instructor Demonstration Notebook**

This capstone notebook brings together all concepts from Days 1–4 and implements a complete end-to-end mini project using the classic EMP–DEPT dataset.

**Topics Covered**
- Python-MySQL connection and database setup
- Table creation with primary and foreign keys
- Bulk data insertion using `executemany()`
- Data exploration using `SELECT`
- Business reports: headcount, salary analysis, top earners
- Filtering above-average earners using subqueries
- Reusable Python function for query execution
- Final consolidated summary report

---

## Project Overview

This mini project builds a self-contained **Employee Management Report System** using Python and MySQL.

**Project Steps:**
1. Connect to MySQL Server
2. Create `employee_db` with `dept` and `emp` tables
3. Populate tables with sample EMP–DEPT data
4. Explore the raw data
5. Generate business reports (headcount, salary, top earners, above-average filter)
6. Build a reusable Python function for executing queries
7. Print a final consolidated summary report

**Schema Overview:**

| Table | Columns |
|-------|---------|
| `dept` | `deptno`, `dname`, `loc` |
| `emp`  | `empno`, `ename`, `job`, `mgr`, `hiredate`, `sal`, `comm`, `deptno` |

---

## 1. Connect to MySQL Server

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="yourpassword"  # replace with your MySQL password
)

cursor = conn.cursor()
print("Connected to MySQL Server successfully!")

## 2. Create Database and Tables

We create the `employee_db` database and two tables:
- **`dept`** — stores department information
- **`emp`** — stores employee details with a foreign key to `dept`

In [ ]:
# Create and select database
cursor.execute("CREATE DATABASE IF NOT EXISTS employee_db")
cursor.execute("USE employee_db")
print("Database 'employee_db' selected!")

# Create dept table
cursor.execute("DROP TABLE IF EXISTS emp")
cursor.execute("DROP TABLE IF EXISTS dept")

cursor.execute("""
CREATE TABLE dept (
    deptno INT PRIMARY KEY,
    dname  VARCHAR(50),
    loc    VARCHAR(50)
)
""")
print("Table 'dept' created!")

# Create emp table with foreign key
cursor.execute("""
CREATE TABLE emp (
    empno    INT PRIMARY KEY,
    ename    VARCHAR(50),
    job      VARCHAR(50),
    mgr      INT,
    hiredate DATE,
    sal      DECIMAL(10, 2),
    comm     DECIMAL(10, 2),
    deptno   INT,
    FOREIGN KEY (deptno) REFERENCES dept(deptno)
)
""")
print("Table 'emp' created!")

## 3. Insert Sample Data

We insert 4 departments and 14 employees using `executemany()` for efficient bulk inserts.

In [ ]:
# Insert department records
dept_data = [
    (10, 'ACCOUNTING', 'NEW YORK'),
    (20, 'RESEARCH',   'DALLAS'),
    (30, 'SALES',      'CHICAGO'),
    (40, 'OPERATIONS', 'BOSTON')
]
cursor.executemany("INSERT INTO dept VALUES (%s, %s, %s)", dept_data)
conn.commit()
print(f"{cursor.rowcount} department records inserted!")

# Insert employee records
emp_data = [
    (7369, 'SMITH',  'CLERK',     7902, '1980-12-17', 800.00,  None,    20),
    (7499, 'ALLEN',  'SALESMAN',  7698, '1981-02-20', 1600.00, 300.00,  30),
    (7521, 'WARD',   'SALESMAN',  7698, '1981-02-22', 1250.00, 500.00,  30),
    (7566, 'JONES',  'MANAGER',   7839, '1981-04-02', 2975.00, None,    20),
    (7654, 'MARTIN', 'SALESMAN',  7698, '1981-09-28', 1250.00, 1400.00, 30),
    (7698, 'BLAKE',  'MANAGER',   7839, '1981-05-01', 2850.00, None,    30),
    (7782, 'CLARK',  'MANAGER',   7839, '1981-06-09', 2450.00, None,    10),
    (7788, 'SCOTT',  'ANALYST',   7566, '1987-07-13', 3000.00, None,    20),
    (7839, 'KING',   'PRESIDENT', None, '1981-11-17', 5000.00, None,    10),
    (7844, 'TURNER', 'SALESMAN',  7698, '1981-09-08', 1500.00, 0.00,    30),
    (7876, 'ADAMS',  'CLERK',     7788, '1987-07-13', 1100.00, None,    20),
    (7900, 'JAMES',  'CLERK',     7698, '1981-12-03', 950.00,  None,    30),
    (7902, 'FORD',   'ANALYST',   7566, '1981-12-03', 3000.00, None,    20),
    (7934, 'MILLER', 'CLERK',     7782, '1982-01-23', 1300.00, None,    10)
]

emp_insert_query = """
INSERT INTO emp (empno, ename, job, mgr, hiredate, sal, comm, deptno)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""
cursor.executemany(emp_insert_query, emp_data)
conn.commit()
print(f"{cursor.rowcount} employee records inserted!")

## 4. Data Exploration

Verify the inserted data with basic `SELECT` queries on both tables.

In [ ]:
# View all departments
cursor.execute("SELECT * FROM dept")
print("=== DEPT Table ===")
for row in cursor.fetchall():
    print(row)

# View all employees
cursor.execute("SELECT empno, ename, job, sal, deptno FROM emp")
print("\n=== EMP Table (selected columns) ===")
for row in cursor.fetchall():
    print(row)

## 5. Report 1 — Department Headcount

Count the number of employees in each department using `GROUP BY` and `JOIN`.

In [ ]:
cursor.execute("""
SELECT d.dname, COUNT(e.empno) AS headcount
FROM dept d
LEFT JOIN emp e ON d.deptno = e.deptno
GROUP BY d.dname
ORDER BY headcount DESC
""")

print("--- Report 1: Department Headcount ---")
print(f"{'Department':<15} {'Headcount':>10}")
print("-" * 27)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>10}")

## 6. Report 2 — Salary Analysis by Department

Compute total salary, average salary, minimum salary, and maximum salary grouped by department.

In [ ]:
cursor.execute("""
SELECT d.dname,
       SUM(e.sal)            AS total_sal,
       ROUND(AVG(e.sal), 2)  AS avg_sal,
       MIN(e.sal)            AS min_sal,
       MAX(e.sal)            AS max_sal
FROM dept d
JOIN emp e ON d.deptno = e.deptno
GROUP BY d.dname
ORDER BY total_sal DESC
""")

print("--- Report 2: Salary Analysis by Department ---")
print(f"{'Department':<15} {'Total':>10} {'Average':>10} {'Min':>8} {'Max':>8}")
print("-" * 55)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {float(row[1]):>10.2f} {float(row[2]):>10.2f} {float(row[3]):>8.2f} {float(row[4]):>8.2f}")

## 7. Report 3 — Top Earning Employees

List the top 5 highest-paid employees with their department name using a `JOIN` and `ORDER BY`.

In [ ]:
cursor.execute("""
SELECT e.ename, e.job, e.sal, d.dname
FROM emp e
JOIN dept d ON e.deptno = d.deptno
ORDER BY e.sal DESC
LIMIT 5
""")

print("--- Report 3: Top 5 Earning Employees ---")
print(f"{'Name':<10} {'Job':<12} {'Salary':>10} {'Department':<15}")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<10} {row[1]:<12} {float(row[2]):>10.2f} {row[3]:<15}")

## 8. Report 4 — Employees Above Average Salary

Use a subquery to identify employees earning more than the company-wide average salary.

In [ ]:
cursor.execute("""
SELECT e.ename, e.job, e.sal
FROM emp e
WHERE e.sal > (SELECT AVG(sal) FROM emp)
ORDER BY e.sal DESC
""")

rows = cursor.fetchall()
print("--- Report 4: Employees Above Average Salary ---")
print(f"{'Name':<10} {'Job':<12} {'Salary':>10}")
print("-" * 35)
for row in rows:
    print(f"{row[0]:<10} {row[1]:<12} {float(row[2]):>10.2f}")
print(f"\nTotal: {len(rows)} employees earn above average.")

## 9. Reusable Query Function

Define a Python helper function that accepts a SQL query string, executes it, and prints results with a header label.  
This pattern improves code reuse and reduces repetition in real-world scripts.

In [ ]:
def run_report(cursor, title, query):
    """Execute a SQL query and print results with a formatted title."""
    cursor.execute(query)
    rows = cursor.fetchall()
    print(f"\n{'=' * 50}")
    print(f"  {title}")
    print(f"{'=' * 50}")
    if rows:
        for row in rows:
            print("  " + " | ".join(str(col) for col in row))
    else:
        print("  No records found.")
    return rows


# Example: job-wise average salary using the reusable function
run_report(
    cursor,
    "Job-wise Average Salary",
    """
    SELECT job, ROUND(AVG(sal), 2) AS avg_sal
    FROM emp
    GROUP BY job
    ORDER BY avg_sal DESC
    """
)

## 10. Final Summary Report

Print a consolidated summary that combines overall statistics and a full employee roster with department names.

In [ ]:
# --- Overall statistics ---
cursor.execute("SELECT COUNT(*), ROUND(AVG(sal), 2), MIN(sal), MAX(sal) FROM emp")
stats = cursor.fetchone()

print("=" * 60)
print("       EMPLOYEE MANAGEMENT SYSTEM — FINAL REPORT")
print("=" * 60)
print(f"  Total Employees  : {stats[0]}")
print(f"  Average Salary   : {float(stats[1]):>10.2f}")
print(f"  Lowest  Salary   : {float(stats[2]):>10.2f}")
print(f"  Highest Salary   : {float(stats[3]):>10.2f}")
print("=" * 60)

# --- Full employee roster ---
cursor.execute("""
SELECT e.empno, e.ename, e.job, e.sal, d.dname, d.loc
FROM emp e
JOIN dept d ON e.deptno = d.deptno
ORDER BY d.dname, e.sal DESC
""")

print(f"\n{'EmpNo':<8} {'Name':<10} {'Job':<12} {'Salary':>9} {'Dept':<12} {'Location'}")
print("-" * 65)
for row in cursor.fetchall():
    print(f"{row[0]:<8} {row[1]:<10} {row[2]:<12} {float(row[3]):>9.2f} {row[4]:<12} {row[5]}")
print("=" * 60)
print("  End of Report")
print("=" * 60)

---
## Practice Tasks

1. **Exercise 1:** Modify Report 1 to show only departments with more than 3 employees (use `HAVING`).
2. **Exercise 2:** Add a `comm` (commission) column to Report 2 showing total commission per department.
3. **Exercise 3:** List all `CLERK` and `SALESMAN` employees using `WHERE job IN (...)`.
4. **Exercise 4:** Find employees who report to `KING` (empno 7839) using a subquery on the `mgr` column.
5. **Exercise 5:** Extend the reusable `run_report()` function to also display the column header names from `cursor.description`.

---
## Conclusion

This notebook demonstrated a complete end-to-end Python + SQL mini project:
- Established a MySQL connection and created a structured schema from Python
- Performed bulk inserts using `executemany()` for efficiency
- Generated four distinct business reports using `JOIN`, `GROUP BY`, aggregate functions, and subqueries
- Built a reusable Python function to eliminate repetitive query-and-print patterns
- Produced a final consolidated summary report combining statistics and a full employee roster

This mini project consolidates all SQL and Python integration skills from Days 1 through 4 of the training.

---
### References
MySQL Documentation: https://dev.mysql.com/doc/

MySQL Connector Python: https://dev.mysql.com/doc/connector-python/en/

Books:
- Learning SQL - Alan Beaulieu
- SQL in 10 Minutes - Ben Forta
- Database System Concepts - Silberschatz, Korth, and Sudarshan

*Notebook generated for SQL and MySQL training (2026).*

---
## Training Workflow Status

| Day | Focus | Status |
|-----|-------|--------|
| **Day 1** | SQL Foundations and Python Connectivity | Complete |
| **Day 2** | Querying and Filtering in SQL | Complete |
| **Day 3** | Aggregations, Joins, and Subqueries | Complete |
| **Day 4** | Data Analysis Using SQL | Complete |
| **Day 5** | Mini Project: Python and SQL Integration | Complete |

---

#### **Author**

**Prakash Ukhalkar**  
Assistant Professor (MCA)  
Researcher in Data Science and Machine Learning  
[![GitHub](https://img.shields.io/badge/GitHub-prakash--ukhalkar-blue?style=flat&logo=github)](https://github.com/prakash-ukhalkar)

---

<div align="center">
  <sub>Built with care for SQL learners and data practitioners</sub>
</div>

In [ ]:
# Cleanup
cursor.close()
conn.close()
print("Connection closed.")